# 02. 탐색적 데이터 분석 (EDA) & 피처 엔지니어링

## 목표
1. 가격지수 추이 시각화 (6개구 비교)
2. 금리·KOSPI와 가격지수 상관관계 분석
3. 정상성(Stationarity) 검정 → ARIMA 차분 차수 결정
4. lag/rolling 피처 생성 → ML 모델 학습용 데이터프레임 저장

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import seaborn as sns
from pathlib import Path
from statsmodels.tsa.stattools import adfuller, kpss

plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['figure.figsize'] = (14, 5)

DATA_RAW  = Path('../data/raw')
DATA_PROC = Path('../data/processed')
DATA_PROC.mkdir(parents=True, exist_ok=True)

TARGET_DISTRICTS = ['노원구', '은평구', '서대문구', '서초구', '강남구', '송파구']

combined = pd.read_csv(DATA_RAW / 'combined_weekly.csv', index_col='date', parse_dates=True)
print(combined.shape)
combined.head()

## 1. 아파트 매매가격지수 추이

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 8))
colors = plt.cm.tab10.colors

for ax, district, color in zip(axes.flat, TARGET_DISTRICTS, colors):
    if district in combined.columns:
        combined[district].plot(ax=ax, color=color, linewidth=1.5)
    ax.set_title(district, fontsize=13)
    ax.set_xlabel('')
    ax.set_ylabel('가격지수')
    ax.grid(alpha=0.3)

plt.suptitle('서울 6개구 아파트 매매가격지수 추이', fontsize=15, y=1.02)
plt.tight_layout()
plt.savefig(DATA_PROC / 'price_trend.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# 6개구 한 그래프에 비교
fig, ax = plt.subplots(figsize=(14, 5))
for district, color in zip(TARGET_DISTRICTS, colors):
    if district in combined.columns:
        combined[district].plot(ax=ax, label=district, linewidth=1.5)
ax.legend(loc='upper left')
ax.set_title('6개구 아파트 매매가격지수 비교')
ax.set_ylabel('가격지수')
ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

## 2. 금융 변수와 상관관계

In [ ]:
fin_cols = [c for c in ['kospi', 'usd_krw', 'base_rate', 'mortgage_rate'] if c in combined.columns]

fig, axes = plt.subplots(len(fin_cols), 1, figsize=(14, 4 * len(fin_cols)))
if len(fin_cols) == 1:
    axes = [axes]

for ax, col in zip(axes, fin_cols):
    ax2 = ax.twinx()
    combined['강남구'].plot(ax=ax, color='steelblue', label='강남구', linewidth=1.5)
    combined[col].plot(ax=ax2, color='tomato', label=col, linewidth=1.2, linestyle='--')
    ax.set_ylabel('강남구 가격지수', color='steelblue')
    ax2.set_ylabel(col, color='tomato')
    ax.set_title(f'강남구 vs {col}')
    lines1, labels1 = ax.get_legend_handles_labels()
    lines2, labels2 = ax2.get_legend_handles_labels()
    ax.legend(lines1 + lines2, labels1 + labels2, loc='upper left')
    ax.grid(alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# 피어슨 상관계수 히트맵
all_cols = TARGET_DISTRICTS + fin_cols
existing = [c for c in all_cols if c in combined.columns]
corr = combined[existing].corr()

plt.figure(figsize=(12, 8))
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, annot=True, fmt='.2f', cmap='RdBu_r', center=0,
            mask=mask, square=True, linewidths=0.5)
plt.title('변수 간 상관계수 히트맵')
plt.tight_layout()
plt.show()

## 3. 정상성 검정 (ADF Test)

ARIMA 차분 차수(d) 결정을 위해 ADF 검정 실시  
귀무가설: 단위근 존재 (비정상) → p < 0.05 이면 정상

In [ ]:
def adf_test(series, name=''):
    result = adfuller(series.dropna())
    return {
        '변수': name,
        'ADF Statistic': round(result[0], 4),
        'p-value': round(result[1], 4),
        '정상성': '정상 ✓' if result[1] < 0.05 else '비정상 ✗'
    }


rows = []
for d in TARGET_DISTRICTS:
    if d in combined.columns:
        rows.append(adf_test(combined[d], d))
        rows.append(adf_test(combined[d].diff(), f'{d} (1차차분)'))

adf_df = pd.DataFrame(rows)
print(adf_df.to_string(index=False))

## 4. 피처 엔지니어링

ML 모델(XGBoost)을 위한 lag/rolling 피처 생성

In [ ]:
def build_ml_dataset(combined_df, target_districts, fin_cols):
    """구별 wide 형식 → 구 ID 포함 long 형식으로 변환 후 피처 생성."""
    records = []

    for district in target_districts:
        if district not in combined_df.columns:
            continue

        df = combined_df[[district] + fin_cols].copy()
        df = df.rename(columns={district: 'price_index'})
        df['district'] = district
        df['is_gangnam'] = int(district in ['서초구', '강남구', '송파구'])

        # lag 피처 (1, 2, 4, 8, 12, 26주 전)
        for lag in [1, 2, 4, 8, 12, 26]:
            df[f'price_lag{lag}'] = df['price_index'].shift(lag)

        # rolling 통계 (4주, 8주, 26주)
        for w in [4, 8, 26]:
            df[f'price_roll_mean{w}'] = df['price_index'].shift(1).rolling(w).mean()
            df[f'price_roll_std{w}']  = df['price_index'].shift(1).rolling(w).std()

        # 가격 변화율
        df['price_chg1'] = df['price_index'].pct_change(1)
        df['price_chg4'] = df['price_index'].pct_change(4)

        # 금융 변수 lag (선행효과: 2~8주 전 금리가 현재 가격에 영향)
        for col in fin_cols:
            for lag in [2, 4, 8]:
                df[f'{col}_lag{lag}'] = df[col].shift(lag)

        # 전세/매매 비율 (갭투자 지표)
        jeonse_col = district + '_전세'
        if jeonse_col in combined_df.columns:
            df['jeonse_ratio'] = combined_df[jeonse_col] / combined_df[district]

        # 시간 피처
        df['month'] = df.index.month
        df['week_of_year'] = df.index.isocalendar().week.astype(int)

        records.append(df)

    ml_df = pd.concat(records).sort_index()
    ml_df['district_code'] = pd.Categorical(ml_df['district']).codes
    return ml_df


fin_cols = [c for c in ['kospi', 'usd_krw', 'base_rate', 'mortgage_rate'] if c in combined.columns]
ml_df = build_ml_dataset(combined, TARGET_DISTRICTS, fin_cols)

print(f'ML 데이터셋: {ml_df.shape}')
print(f'피처 수: {len(ml_df.columns)}')
ml_df.head()

In [ ]:
# 저장 (결측치 포함 행 제거)
ml_clean = ml_df.dropna()
ml_clean.to_csv(DATA_PROC / 'ml_features.csv')
print(f'저장 완료: ml_features.csv ({ml_clean.shape})')

# 피처 중요도 확인용: 결측치 현황
print('\n결측치 비율 (상위 10개):')
print((ml_df.isnull().mean() * 100).sort_values(ascending=False).head(10).round(1))